In [7]:
import numpy as np
from math import log, sqrt

def compute_threshold(log2_dfp):
    """
    Compute the threshold corresponding to the log2 decryption failure probability.
    """
    ln2 = log(2)
    return sqrt(2 * ln2 * (1 - log2_dfp))


def classical_attack_complexity(alpha, log2_dfp, dimension):
    """
    Compute the DFA attack complexity (in log2 scale).
    """

    prob1 = (
        dimension * log(alpha)
        + (dimension / 2) * (1 - alpha**2)
    )
    classical_complexity = -prob1 / log(2)
    quantum_complexity = classical_complexity/2

    # Complexity due to decryption failures
    threshold = compute_threshold(log2_dfp)
    beta = threshold / alpha

    prob2 = log(2) - beta**2 / 2
    failure_complexity = -prob2 / log(2)

    classical_complexity += failure_complexity
    quantum_complexity += failure_complexity

    # print(f"scale: {alpha}, threshold : {threshold}, beta : {beta}")

    return alpha, classical_complexity, quantum_complexity


# ---------------------------------------------------------------------
# Parameter Sweep
# ---------------------------------------------------------------------

alpha_values = np.arange(1.00, 1.50, 0.05)
# alpha_values = [1.1]

variants = {
    "CCAKEM-512": {
        "rank": 2,
        "n": 256,
        "log2_dfp": -158.9,
    },
    "CCAKEM-768": {
        "rank": 3,
        "n": 256,
        "log2_dfp": -165.6,
    },
    "CCAKEM-1024": {
        "rank": 4,
        "n": 256,
        "log2_dfp": -181.8,
    },
}

complexities = {}

for variant_name, params in variants.items():
    dimension = 2 * params["rank"] * params["n"]

    results = [
        classical_attack_complexity(
            alpha,
            params["log2_dfp"],
            dimension,
        )
        for alpha in alpha_values
    ]

    complexities[variant_name] = results

    print(f"\n=============== {variant_name} ====================")
    print(f"{'Alpha':>6} {'Classical':>15} {'Quantum':>15}")
    print("-" * 40)

    for alpha, classical, quantum in complexities[variant_name]:
        print(f"{alpha:6.2f} {classical:15.2f} {quantum:15.2f}")




=============== CCAKEM-512 ====================
 Alpha       Classical         Quantum
----------------------------------------
  1.00          158.90          158.90
  1.05          147.67          145.85
  1.10          145.46          138.31
  1.15          151.65          135.78
  1.20          165.70          137.87
  1.25          187.18          144.26
  1.30          215.69          154.66
  1.35          250.93          168.84
  1.40          292.62          186.60
  1.45          340.51          207.78

=============== CCAKEM-768 ====================
 Alpha       Classical         Quantum
----------------------------------------
  1.00          165.60          165.60
  1.05          155.56          152.84
  1.10          158.16          147.42
  1.15          172.59          148.78
  1.20          198.19          156.44
  1.25          234.39          170.01
  1.30          280.70          189.14
  1.35          336.71          213.56
  1.40          402.05          243.03
 